# Data — Build Per-Persona Subsets

Filters PersonaMem-v2 to personas with enough training and held-out examples and writes one subset per persona. Each persona becomes a client in the federated setting.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [2]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub pandas pyarrow

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"
SUBSETS_DIR.mkdir(parents=True, exist_ok=True)
print("Subsets will be saved to:", SUBSETS_DIR.resolve())

Mounted at /content/drive
Subsets will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets


In [3]:
import json
import random

import pandas as pd
from datasets import load_dataset

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42

MIN_TRAIN_ROWS = 20
MIN_VAL_ROWS = 4

NUM_SUBSETS = 3
ENFORCE_EQUAL = True

random.seed(SEED)

def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)

In [4]:
train_df = load_split("train_text").to_pandas()
val_df = load_split("val_text").to_pandas()
print(f"train_text rows: {len(train_df):,} | val_text rows: {len(val_df):,}")
print(f"unique personas in train: {train_df['persona_id'].nunique():,}")
print(f"unique personas in val:   {val_df['persona_id'].nunique():,}")

README.md:   0%|          | 0.00/15.9k [00:00<?, ?B/s]

benchmark/multimodal/benchmark.csv: reconstructing file:   0%|          |  0.00B / 42.4MB            

benchmark/multimodal/benchmark.csv: downloading bytes:           |  0.00B            

benchmark/multimodal/train.csv: reconstructing file:   0%|          |  0.00B /  161MB            

benchmark/multimodal/train.csv: downloading bytes:           |  0.00B            

...

Generating val_multimodal split:   0%|          | 0/2111 [00:00<?, ? examples/s]

Generating benchmark_text split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating train_text split:   0%|          | 0/18549 [00:00<?, ? examples/s]

Generating val_text split:   0%|          | 0/2061 [00:00<?, ? examples/s]

train_text rows: 18,549 | val_text rows: 2,061
unique personas in train: 800
unique personas in val:   735


In [5]:
train_counts = train_df.groupby("persona_id").size()
val_counts = val_df.groupby("persona_id").size()

eligible = [
    pid
    for pid in train_counts.index
    if train_counts[pid] > MIN_TRAIN_ROWS
    and pid in val_counts.index
    and val_counts[pid] >= MIN_VAL_ROWS
]
eligible = sorted(eligible)

print(f"Eligible personas (>{MIN_TRAIN_ROWS} train & >={MIN_VAL_ROWS} val): {len(eligible)}")
if eligible:
    tr = train_counts[eligible]
    va = val_counts[eligible]
    print(f"  train rows  min/mean/max: {tr.min()}/{tr.mean():.1f}/{tr.max()}")
    print(f"  val rows    min/mean/max: {va.min()}/{va.mean():.1f}/{va.max()}")

Eligible personas (>20 train & >=4 val): 150
  train rows  min/mean/max: 21/25.8/38
  val rows    min/mean/max: 4/4.8/8


In [6]:
rng = random.Random(SEED)
shuffled = eligible[:]
rng.shuffle(shuffled)

per_subset = len(shuffled) // NUM_SUBSETS
if per_subset == 0:
    raise ValueError(
        f"Only {len(shuffled)} eligible personas - not enough for {NUM_SUBSETS} subsets."
    )

if ENFORCE_EQUAL:
    used = shuffled[: per_subset * NUM_SUBSETS]
    dropped = shuffled[per_subset * NUM_SUBSETS :]
    subsets = {
        f"subset_{i}": sorted(used[i * per_subset : (i + 1) * per_subset])
        for i in range(NUM_SUBSETS)
    }
else:
    dropped = []
    subsets = {f"subset_{i}": [] for i in range(NUM_SUBSETS)}
    for idx, pid in enumerate(shuffled):
        subsets[f"subset_{idx % NUM_SUBSETS}"].append(pid)
    subsets = {name: sorted(ids) for name, ids in subsets.items()}

for name, ids in subsets.items():
    print(f"{name}: {len(ids)} personas | first 5 = {ids[:5]}")
print(f"dropped (leftover): {len(dropped)} -> {dropped}")

subset_0: 50 personas | first 5 = [6, 18, 95, 105, 148]
subset_1: 50 personas | first 5 = [14, 33, 73, 80, 93]
subset_2: 50 personas | first 5 = [9, 41, 51, 56, 57]
dropped (leftover): 0 -> []


In [7]:
manifest = {
    "dataset": DATASET_NAME,
    "config": CONFIG,
    "seed": SEED,
    "min_train_rows": MIN_TRAIN_ROWS,
    "min_val_rows": MIN_VAL_ROWS,
    "num_subsets": NUM_SUBSETS,
    "enforce_equal": ENFORCE_EQUAL,
    "per_subset_size": per_subset if ENFORCE_EQUAL else None,
    "num_eligible": len(eligible),
    "dropped_personas": [int(p) for p in dropped],
    "subsets": {name: [int(p) for p in ids] for name, ids in subsets.items()},
}

with open(SUBSETS_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

for name, ids in subsets.items():
    subset_dir = SUBSETS_DIR / name
    subset_dir.mkdir(parents=True, exist_ok=True)

    sub_train = train_df[train_df["persona_id"].isin(ids)].reset_index(drop=True)
    sub_val = val_df[val_df["persona_id"].isin(ids)].reset_index(drop=True)

    sub_train.to_parquet(subset_dir / "train.parquet", index=False)
    sub_val.to_parquet(subset_dir / "val.parquet", index=False)
    with open(subset_dir / "personas.json", "w", encoding="utf-8") as f:
        json.dump([int(p) for p in ids], f, indent=2)

    print(
        f"saved {name}: {len(ids)} personas | "
        f"train {len(sub_train):,} rows | val {len(sub_val):,} rows -> {subset_dir}"
    )

print("\nManifest saved to:", (SUBSETS_DIR / "manifest.json").resolve())

saved subset_0: 50 personas | train 1,267 rows | val 230 rows -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/subset_0
saved subset_1: 50 personas | train 1,302 rows | val 240 rows -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/subset_1
saved subset_2: 50 personas | train 1,301 rows | val 244 rows -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/subset_2

Manifest saved to: /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/manifest.json


In [8]:
def load_manifest(subsets_dir=SUBSETS_DIR):
    with open(Path(subsets_dir) / "manifest.json", "r", encoding="utf-8") as f:
        return json.load(f)


def list_subsets(subsets_dir=SUBSETS_DIR):
    return sorted(load_manifest(subsets_dir)["subsets"].keys())


def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Return (persona_ids, train_df, val_df) for a saved subset."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"No such subset folder: {subset_dir}")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = json.load(f)
    sub_train = pd.read_parquet(subset_dir / "train.parquet")
    sub_val = pd.read_parquet(subset_dir / "val.parquet")
    return persona_ids, sub_train, sub_val

In [9]:
meta = load_manifest()
print("Subsets available:", list_subsets())
print("Selection rule: > %d train rows & >= %d val rows" % (meta["min_train_rows"], meta["min_val_rows"]))

for name in list_subsets():
    ids, tr, va = load_subset(name)
    assert set(tr["persona_id"]) <= set(ids)
    assert set(va["persona_id"]) <= set(ids)
    print(f"{name}: {len(ids)} personas | train {len(tr):,} rows | val {len(va):,} rows")

Subsets available: ['subset_0', 'subset_1', 'subset_2']
Selection rule: > 20 train rows & >= 4 val rows
subset_0: 50 personas | train 1,267 rows | val 230 rows
subset_1: 50 personas | train 1,302 rows | val 240 rows
subset_2: 50 personas | train 1,301 rows | val 244 rows


In [10]:
from huggingface_hub import hf_hub_download
import re, time

NEG_HISTORY_VARIANT = "32k"
NEG_SIM_METHOD      = "tfidf"
NEG_MIN_TURNS       = 2
NEG_TOPK_OFFTOPIC   = 3
NEG_SEED            = SEED
NEG_COL             = "negative_conversation_snippet"

HISTORY_CACHE_DIR = PP_ROOT / "chat_history_cache"
HISTORY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LINK_COL = "chat_history_32k_link" if NEG_HISTORY_VARIANT == "32k" else "chat_history_128k_link"
print("Mining negatives from", LINK_COL, "| similarity:", NEG_SIM_METHOD)

Mining negatives from chat_history_32k_link | similarity: tfidf


In [11]:
def download_persona_history(link, retries=4):
    """Download & parse one persona's chat history -> list of {role, content} turns
    (system turn dropped so negatives never leak the persona system prompt)."""
    last_err = None
    for attempt in range(retries):
        try:
            local = hf_hub_download(
                repo_id=DATASET_NAME, filename=link, repo_type="dataset",
                cache_dir=str(HISTORY_CACHE_DIR),
            )
            with open(local, "r", encoding="utf-8") as f:
                data = json.load(f)
            turns = data["chat_history"] if isinstance(data, dict) else data
            return [t for t in turns if isinstance(t, dict) and t.get("role") != "system"]
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to download {link}: {last_err}")

In [12]:
def _norm(s):
    return re.sub(r"\s+", " ", str(s)).strip()

def parse_turns(raw):
    """Parse a snippet (JSON string / list) into a list of {role, content} turns."""
    if isinstance(raw, list):
        return raw
    if raw is None or (isinstance(raw, float)):
        return []
    try:
        val = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        try:
            import ast
            val = ast.literal_eval(raw)
        except Exception:
            return []
    return val if isinstance(val, list) else []

def turns_text(turns):
    return " ".join(f"{t.get('role','')}: {t.get('content','')}" for t in turns if isinstance(t, dict))

def _locate_span(hist, pos_turns):
    """Return (start, end) of the positive snippet inside history, or None."""
    hn = [(t.get("role"), _norm(t.get("content"))) for t in hist]
    pn = [(t.get("role"), _norm(t.get("content"))) for t in pos_turns]
    k = len(pn)
    if k == 0 or k > len(hn):
        return None
    for i in range(len(hn) - k + 1):
        if hn[i:i + k] == pn:
            return (i, i + k)
    for i in range(len(hn) - k + 1):
        if hn[i][1] == pn[0][1] and hn[i + k - 1][1] == pn[-1][1]:
            return (i, i + k)
    return None

_TOK = re.compile(r"\w+", re.UNICODE)
def _toks(t):
    return set(m.group().lower() for m in _TOK.finditer(t))
def _jaccard(a, b):
    return len(a & b) / max(1, len(a | b))

def _similarities(pos_text, cand_texts):
    """Similarity of each candidate window to the positive snippet (lower = more off-topic)."""
    if NEG_SIM_METHOD == "tfidf":
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.metrics.pairwise import cosine_similarity
            vec = TfidfVectorizer().fit([pos_text] + cand_texts)
            mats = vec.transform([pos_text] + cand_texts)
            return list(cosine_similarity(mats[0:1], mats[1:]).ravel())
        except Exception:
            pass
    pt = _toks(pos_text)
    return [_jaccard(pt, _toks(c)) for c in cand_texts]

In [13]:
def choose_negative_snippet(hist, pos_turns, rng):
    """Pick the most off-topic window of the persona's OWN history as negative context.
    Returns (json_string_of_turns, meta) or (None, {}) if no history."""
    if not hist:
        return None, {}
    k = max(NEG_MIN_TURNS, len(pos_turns) or NEG_MIN_TURNS)
    k = min(k, len(hist))
    span = _locate_span(hist, pos_turns) if pos_turns else None
    step = max(1, k)

    def overlaps(a, b):
        return span is not None and a < span[1] and span[0] < b

    windows = [(s, s + k) for s in range(0, len(hist) - k + 1, step) if not overlaps(s, s + k)]
    if not windows:
        windows = [(s, s + k) for s in range(0, len(hist) - k + 1)] or [(0, len(hist))]
    cand_texts = [turns_text(hist[s:e]) for s, e in windows]
    pos_text = turns_text(pos_turns) if pos_turns else cand_texts[0]
    sims = _similarities(pos_text, cand_texts)
    order = sorted(range(len(windows)), key=lambda i: sims[i])
    idx = rng.choice(order[: min(NEG_TOPK_OFFTOPIC, len(order))])
    s, e = windows[idx]
    meta = {
        "negative_snippet_distance_turns": abs(s - span[0]) if span else -1,
        "negative_snippet_similarity": float(sims[idx]),
        "negative_snippet_source": "own_history",
    }
    return json.dumps(hist[s:e], ensure_ascii=False), meta

In [14]:
from tqdm.auto import tqdm

neg_rng = random.Random(NEG_SEED)
neg_preview = []

for name, ids in subsets.items():
    subset_dir = SUBSETS_DIR / name
    tr = pd.read_parquet(subset_dir / "train.parquet")
    if LINK_COL not in tr.columns:
        raise KeyError(f"{LINK_COL} missing from {name}/train.parquet")

    hist_by_persona = {}
    for pid, grp in tr.groupby("persona_id"):
        link = grp[LINK_COL].iloc[0]
        try:
            hist_by_persona[pid] = download_persona_history(link)
        except Exception as e:
            print(f"  [warn] {name} persona {pid}: {e}")
            hist_by_persona[pid] = []

    neg_col, dist_col, sim_col, src_col = [], [], [], []
    n_ok = 0
    for _, row in tqdm(tr.iterrows(), total=len(tr), desc=f"{name} negatives"):
        hist = hist_by_persona.get(row["persona_id"], [])
        pos_turns = parse_turns(row.get("related_conversation_snippet"))
        neg_json, meta = choose_negative_snippet(hist, pos_turns, neg_rng)
        neg_col.append(neg_json)
        dist_col.append(meta.get("negative_snippet_distance_turns", -1))
        sim_col.append(meta.get("negative_snippet_similarity", float("nan")))
        src_col.append(meta.get("negative_snippet_source", "none") if neg_json else "none")
        if neg_json:
            n_ok += 1
            if len(neg_preview) < 20:
                neg_preview.append({
                    "subset": name, "persona_id": int(row["persona_id"]),
                    "pos_preview": turns_text(pos_turns)[:160],
                    "neg_preview": turns_text(parse_turns(neg_json))[:160],
                    "similarity": meta.get("negative_snippet_similarity"),
                })

    tr[NEG_COL] = neg_col
    tr["negative_snippet_distance_turns"] = dist_col
    tr["negative_snippet_similarity"] = sim_col
    tr["negative_snippet_source"] = src_col
    tr.to_parquet(subset_dir / "train.parquet", index=False)
    print(f"{name}: negatives for {n_ok}/{len(tr)} rows -> {subset_dir / 'train.parquet'}")

with open(SUBSETS_DIR / "negative_context_preview.json", "w", encoding="utf-8") as f:
    json.dump(neg_preview, f, indent=2, ensure_ascii=False)
print("\nPreview saved ->", SUBSETS_DIR / "negative_context_preview.json")

chat_history_250913_163134_persona6.json:   0%|          | 0.00/182k [00:00<?, ?B/s]

(…)hat_history_250913_163134_persona18.json:   0%|          | 0.00/171k [00:00<?, ?B/s]

(…)hat_history_250913_163134_persona95.json:   0%|          | 0.00/179k [00:00<?, ?B/s]

(…)at_history_250913_163134_persona105.json:   0%|          | 0.00/170k [00:00<?, ?B/s]

(…)at_history_250913_163134_persona148.json:   0%|          | 0.00/182k [00:00<?, ?B/s]

...

(…)at_history_250913_163134_persona960.json:   0%|          | 0.00/170k [00:00<?, ?B/s]

(…)at_history_250913_163134_persona969.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

(…)at_history_250913_163134_persona987.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

subset_2 negatives:   0%|          | 0/1301 [00:00<?, ?it/s]

subset_2: negatives for 1301/1301 rows -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/subset_2/train.parquet

Preview saved -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/negative_context_preview.json


In [15]:
ids, tr, va = load_subset(list_subsets()[0])
neg_cols = [c for c in tr.columns if c.startswith("negative")]
print("New columns:", neg_cols)
cov = tr["negative_snippet_source"].eq("own_history").mean()
print(f"own_history coverage in {list_subsets()[0]}: {cov:.1%}")
print("mean off-topic similarity:", round(float(tr['negative_snippet_similarity'].mean()), 4))
row0 = tr.iloc[0]
print("\nPOSITIVE:", turns_text(parse_turns(row0['related_conversation_snippet']))[:200])
print("\nNEGATIVE:", turns_text(parse_turns(row0[NEG_COL]))[:200])

New columns: ['negative_conversation_snippet', 'negative_snippet_distance_turns', 'negative_snippet_similarity', 'negative_snippet_source']
own_history coverage in subset_0: 100.0%
mean off-topic similarity: 0.0481

POSITIVE: user: What are some good ways to stay calm and focused when work gets really hectic? assistant: When work gets hectic and you start to feel mild anxiety creeping in, try breaking tasks into small mana

NEGATIVE: user: Please forget that I love spontaneous solo travel. assistant: Got it — I’ll forget that you love spontaneous solo travel and won’t take it into account going forward.  

Would you like me to com
